# 05a: Computational efficiency, building the input complexes

**Paper:** Section *Computational Efficiency* (`sec:exp:efficiency`). This notebook builds every complex that 05b–05d time; 05e makes the figures.

Writes `data/efficiency/inputs.npz` with, for every complex `NAME`:
* `NAME/0/x`, `NAME/0/w`: vertex coordinates (float64) and weights;
* `NAME/k/x`, `NAME/k/w`: the k-cells as vertex ids (int64) and their weights, k ≥ 1;
* and a JSON `index` (`NAME -> {dim, kind, source, cells, ...}`).

**Complexes** (25 in total, from 4,300 to about 4 million cells):
* **2D images** of a smooth random field on a disk (Gaussian-filtered noise, the same field at every resolution, 8-bit levels 1–255), at $64^2, 128^2, 256^2, 512^2, 1024^2$:
  * grayscale, as pyECT's weighted Freudenthal triangulation (`img{n}_gray_freudenthal`);
  * binary (the brighter half of the disk), as a cubical complex and as its Freudenthal triangulation (`img{n}_bin_cubical`, `img{n}_bin_freudenthal`).
* **One barley seed**, the median-sized of 60 random founder seeds (`S166/l3_Green_seeds/seed_15_0_p7_d4_t120_o7_e1_g3.tif`), cleaned as in `barley.load_seed`:
  * marching-cubes surface meshes at step sizes 4, 3, 2, 1 (`seed_mesh_step{s}`);
  * the solid voxel complex downsampled by 4, 2, 1, as a cubical complex and as its Freudenthal triangulation (`seed_vox{f}_cubical`, `seed_vox{f}_freudenthal`).

`RECOMPUTE = False` loads the shipped `inputs.npz` and only lists it. Building it needs the extracted CT scans (`00-data-barley.ipynb`) and takes a few minutes on a CPU.

In [ ]:
import glob
import itertools
import json
import os

import numpy as np
import torch
from pyect import weighted_cubical, weighted_freudenthal
from scipy import ndimage
from skimage import measure

import barley as B

RECOMPUTE = False
OUT = B.DATA / "efficiency" / "inputs.npz"

In [ ]:
def describe(index):
    for name, meta in index.items():
        print(f"{name:28s} {meta['total_cells']:>10,d} cells {meta['cells']}")


if OUT.exists() and not RECOMPUTE:
    index = json.loads(bytes(np.load(OUT)["index"]).decode())
    print(f"{OUT.relative_to(B.ROOT)} exists ({len(index)} complexes); set RECOMPUTE = True to rebuild it")
    describe(index)

## Builders

In [ ]:
arrays, index = {}, {}


def store(name, cells, **meta):
    """cells: list of (array, weights) in pyECT order."""
    for k, (x, w) in enumerate(cells):
        arrays[f"{name}/{k}/x"] = np.asarray(x, dtype=np.float64 if k == 0 else np.int64)
        arrays[f"{name}/{k}/w"] = np.asarray(w, dtype=np.float64)
    meta["cells"] = [int(len(w)) for _, w in cells]
    meta["total_cells"] = int(sum(meta["cells"]))
    index[name] = meta
    print(f"{name:28s} {meta['total_cells']:>10,d} cells {meta['cells']}")


def grayscale_image(n, rng):
    """Smooth positive random field on a disk, 8-bit levels 1..255, zero outside."""
    base = ndimage.gaussian_filter(rng.standard_normal((1024, 1024)), 40)
    base = (base - base.min()) / (base.max() - base.min())
    img = ndimage.zoom(base, n / 1024, order=1)
    yy, xx = np.mgrid[:n, :n]
    disk = (xx - (n - 1) / 2) ** 2 + (yy - (n - 1) / 2) ** 2 <= (0.45 * n) ** 2
    return np.where(disk, 1 + np.round(254 * img), 0).astype(np.float32)


def from_pyect(cx):
    return [(cx[k][0].double().numpy() if k == 0 else cx[k][0].numpy(), cx[k][1].double().numpy())
            for k in range(len(cx))]


def voxel_complexes(img, scale):
    """Pixel/voxel-centre cubical complex of a binary image and its Freudenthal
    triangulation (every k-cube split along the monotone lattice paths from its
    lower corner; no new vertices), both with unit weights."""
    n = img.ndim
    UNIT = np.eye(n, dtype=np.int64)
    coords = np.argwhere(img)
    lookup = -np.ones(np.array(img.shape) + 1, dtype=np.int64)
    lookup[tuple(coords.T)] = np.arange(len(coords))

    def ids(offsets):
        out = np.stack([lookup[tuple((coords + o).T)] for o in offsets], 1)
        return out[(out >= 0).all(1)]

    z = np.zeros(n, dtype=np.int64)
    axes_sets = {k: list(itertools.combinations(range(n), k)) for k in range(1, n + 1)}

    # cubical: every k-cube spanned by a set of axes from a lower corner
    cubical = []
    for k in range(1, n + 1):
        cells = []
        for axes in axes_sets[k]:
            offsets = [z + sum((UNIT[a] for a, bit in zip(axes, bits) if bit), z)
                       for bits in itertools.product((0, 1), repeat=k)]
            cells.append(ids(offsets))
        cubical.append(np.concatenate(cells))

    # Freudenthal: every k-cube split along monotone paths from its lower corner
    top = []
    for k in range(1, n + 1):
        for axes in axes_sets[k]:
            for perm in itertools.permutations(axes):
                path, p = [z.copy()], z.copy()
                for a in perm:
                    p = p + UNIT[a]
                    path.append(p.copy())
                # only cubes present in the cubical complex
                all_offsets = [z + sum((UNIT[a] for a, bit in zip(axes, bits) if bit), z)
                               for bits in itertools.product((0, 1), repeat=k)]
                present = np.stack([lookup[tuple((coords + o).T)] for o in all_offsets], 1)
                keep = (present >= 0).all(1)
                simp = np.stack([lookup[tuple((coords + o).T)] for o in path], 1)[keep]
                top.append((k, simp))
    simplices = {k: [] for k in range(1, n + 1)}
    for k, simp in top:
        for j in range(1, k + 1):
            for comb in itertools.combinations(range(k + 1), j + 1):
                simplices[j].append(simp[:, comb])
    freud = [np.unique(np.sort(np.concatenate(simplices[j]), axis=1), axis=0) for j in range(1, n + 1)]

    x = (coords - coords.mean(0)) * scale
    ones = lambda a: np.ones(len(a))
    return ([(x, ones(x))] + [(c, ones(c)) for c in cubical],
            [(x, ones(x))] + [(c, ones(c)) for c in freud])

## Build

In [ ]:
if RECOMPUTE or not OUT.exists():
    # 2D grayscale images; the binary images are added at the end (as in the original run)
    BINARY_IMAGES = {}
    for n in (64, 128, 256, 512, 1024):
        gray = grayscale_image(n, np.random.default_rng(1))   # same field at every resolution
        store(f"img{n}_gray_freudenthal", from_pyect(weighted_freudenthal(torch.from_numpy(gray))), dim=2,
              kind="freudenthal", source="grayscale image", resolution=n)
        BINARY_IMAGES[n] = gray > np.median(gray[gray > 0])

    # the barley seed: median size among 60 random seeds
    rng = np.random.default_rng(0)
    files = sorted(glob.glob(os.path.join(B.SEED_DIR, "*", "*", "*.tif")))
    sample = rng.choice(len(files), 60, replace=False)
    sizes = sorted((int(B.load_seed(files[i]).sum()), files[i]) for i in sample)
    voxels, seed_path = sizes[len(sizes) // 2]
    seed = B.load_seed(seed_path)
    seed = seed[tuple(slice(a.min(), a.max() + 1) for a in np.nonzero(seed))]
    seed_file = os.path.relpath(seed_path, B.SEED_DIR)
    print(f"seed {seed_file}: {voxels:,d} voxels, box {seed.shape}")
    SEED_META = dict(source="barley seed", seed_file=seed_file)
    centre = np.argwhere(seed).mean(0)

    for step in (4, 3, 2, 1):     # surface meshes
        v, f, *_ = measure.marching_cubes(np.pad(seed, 1).astype(np.float32), 0.5, step_size=step)
        v = v.astype(np.float64) - 1 - centre
        f = np.sort(f.astype(np.int64), axis=1)
        e = np.unique(np.concatenate([f[:, [0, 1]], f[:, [0, 2]], f[:, [1, 2]]]), axis=0)
        store(f"seed_mesh_step{step}", [(v, np.ones(len(v))), (e, np.ones(len(e))), (f, np.ones(len(f)))],
              dim=3, kind="mesh", step=step, **SEED_META)

    for factor in (4, 2, 1):      # solid voxel complexes
        img = seed if factor == 1 else ndimage.zoom(seed.astype(np.float32), 1 / factor, order=1) > 0.5
        cub, fr = voxel_complexes(img, factor)
        store(f"seed_vox{factor}_cubical", cub, dim=3, kind="cubical", downsample=factor, **SEED_META)
        store(f"seed_vox{factor}_freudenthal", fr, dim=3, kind="freudenthal", downsample=factor, **SEED_META)
        for label, cx in (("cubical", cub), ("freudenthal", fr)):   # a solid seed: chi = 1
            print(f"    Euler characteristic ({label}): {sum((-1) ** k * len(w) for k, (_, w) in enumerate(cx))}")

    # binary images: cubical complex and its Freudenthal triangulation (pyECT's weighted_freudenthal
    # adds a triangle wherever three pixels are present, a different complex)
    for n, binary in BINARY_IMAGES.items():
        cub, fr = voxel_complexes(binary, 1.0)
        store(f"img{n}_bin_cubical", cub, dim=2, kind="cubical", source="binary image", resolution=n)
        store(f"img{n}_bin_freudenthal", fr, dim=2, kind="freudenthal", source="binary image", resolution=n)
        pyect_cub = from_pyect(weighted_cubical(torch.from_numpy(binary.astype(np.float32))))
        assert [len(w) for _, w in pyect_cub] == [len(w) for _, w in cub], "cubical complex differs from pyECT's"

    arrays["index"] = np.frombuffer(json.dumps(index).encode(), dtype=np.uint8)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(OUT, **arrays)
    print(f"wrote {OUT} ({os.path.getsize(OUT) / 2**20:.0f} MB)")